In [2]:
import os
import pandas as pd

DAISEE_ROOT = "/kaggle/input/datasets/rasangabawanthi/mindbridge-daisee/DAiSEE"

csv_files = []

for root, dirs, files in os.walk(DAISEE_ROOT):
    for file in files:
        if file.lower().endswith(".csv"):
            csv_files.append(
                os.path.join(root, file)
            )

print("CSV files found:", len(csv_files))

for path in csv_files:
    print(path)

CSV files found: 4
/kaggle/input/datasets/rasangabawanthi/mindbridge-daisee/DAiSEE/Labels/TrainLabels.csv
/kaggle/input/datasets/rasangabawanthi/mindbridge-daisee/DAiSEE/Labels/AllLabels.csv
/kaggle/input/datasets/rasangabawanthi/mindbridge-daisee/DAiSEE/Labels/TestLabels.csv
/kaggle/input/datasets/rasangabawanthi/mindbridge-daisee/DAiSEE/Labels/ValidationLabels.csv


In [3]:
import pandas as pd
import os

LABEL_DIR = "/kaggle/input/datasets/rasangabawanthi/mindbridge-daisee/DAiSEE/Labels"

train_labels = pd.read_csv(
    os.path.join(LABEL_DIR, "TrainLabels.csv")
)

val_labels = pd.read_csv(
    os.path.join(LABEL_DIR, "ValidationLabels.csv")
)

test_labels = pd.read_csv(
    os.path.join(LABEL_DIR, "TestLabels.csv")
)

print("Train shape:", train_labels.shape)
print("Validation shape:", val_labels.shape)
print("Test shape:", test_labels.shape)

print("\nColumns:")
print(train_labels.columns.tolist())

print("\nFirst 5 rows:")
display(train_labels.head())

for name, df in [
    ("TRAIN", train_labels),
    ("VALIDATION", val_labels),
    ("TEST", test_labels)
]:
    print(f"\n===== {name} ENGAGEMENT =====")
    
    print("Unique values:")
    print(sorted(df["Engagement"].dropna().unique()))
    
    print("\nCounts:")
    print(df["Engagement"].value_counts().sort_index())
    
    print("\nPercentages:")
    print(
        (df["Engagement"].value_counts(normalize=True)
         .sort_index() * 100)
         .round(2)
    )

Train shape: (5358, 5)
Validation shape: (1429, 5)
Test shape: (1784, 5)

Columns:
['ClipID', 'Boredom', 'Engagement', 'Confusion', 'Frustration ']

First 5 rows:


,ClipID,Boredom,Engagement,Confusion,Frustration
0,1100011002.avi,0,2,0,0
1,1100011003.avi,0,2,0,0
2,1100011004.avi,0,3,0,0
3,1100011005.avi,0,3,0,0
4,1100011006.avi,0,3,0,0



===== TRAIN ENGAGEMENT =====
Unique values:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3)]

Counts:
Engagement
0      34
1     213
2    2617
3    2494
Name: count, dtype: int64

Percentages:
Engagement
0     0.63
1     3.98
2    48.84
3    46.55
Name: proportion, dtype: float64

===== VALIDATION ENGAGEMENT =====
Unique values:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3)]

Counts:
Engagement
0     23
1    143
2    813
3    450
Name: count, dtype: int64

Percentages:
Engagement
0     1.61
1    10.01
2    56.89
3    31.49
Name: proportion, dtype: float64

===== TEST ENGAGEMENT =====
Unique values:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3)]

Counts:
Engagement
0      4
1     84
2    882
3    814
Name: count, dtype: int64

Percentages:
Engagement
0     0.22
1     4.71
2    49.44
3    45.63
Name: proportion, dtype: float64


In [4]:
# =========================================================
# STEP 3 — CREATE DISENGAGEMENT PROXY LABEL
# =========================================================

def add_disengagement_proxy(df):
    df = df.copy()

    # DAiSEE Engagement:
    # 0,1 = low engagement → disengagement proxy
    # 2,3 = engaged

    df["disengaged_label"] = (
        df["Engagement"] <= 1
    ).astype(int)

    return df


train_labels = add_disengagement_proxy(train_labels)
val_labels = add_disengagement_proxy(val_labels)
test_labels = add_disengagement_proxy(test_labels)


for name, df in [
    ("TRAIN", train_labels),
    ("VALIDATION", val_labels),
    ("TEST", test_labels)
]:
    print(f"\n===== {name} =====")

    print(
        df["disengaged_label"]
        .value_counts()
        .sort_index()
    )

    print("\nPercentages:")

    print(
        (
            df["disengaged_label"]
            .value_counts(normalize=True)
            .sort_index() * 100
        ).round(2)
    )


# Training class weight
neg = (
    train_labels["disengaged_label"] == 0
).sum()

pos = (
    train_labels["disengaged_label"] == 1
).sum()

scale_pos_weight = neg / pos

print("\nTrain negatives:", neg)
print("Train positives:", pos)
print(
    "Suggested scale_pos_weight:",
    round(scale_pos_weight, 4)
)


===== TRAIN =====
disengaged_label
0    5111
1     247
Name: count, dtype: int64

Percentages:
disengaged_label
0    95.39
1     4.61
Name: proportion, dtype: float64

===== VALIDATION =====
disengaged_label
0    1263
1     166
Name: count, dtype: int64

Percentages:
disengaged_label
0    88.38
1    11.62
Name: proportion, dtype: float64

===== TEST =====
disengaged_label
0    1696
1      88
Name: count, dtype: int64

Percentages:
disengaged_label
0    95.07
1     4.93
Name: proportion, dtype: float64

Train negatives: 5111
Train positives: 247
Suggested scale_pos_weight: 20.6923


In [5]:
# =========================================================
# STEP 4 — FIND EXISTING VISUAL FEATURE FILES
# =========================================================

import os

search_root = "/kaggle/input"

wanted_files = [
    "daisee_train_visual_features_rollfix.csv",
    "daisee_validation_visual_features.csv",
    "daisee_test_visual_features.csv"
]

found_files = {}

for root, dirs, files in os.walk(search_root):
    for file in files:
        if file in wanted_files:
            full_path = os.path.join(root, file)
            found_files[file] = full_path

for file in wanted_files:
    print(file)
    print("→", found_files.get(file, "NOT FOUND"))
    print()

daisee_train_visual_features_rollfix.csv
→ /kaggle/input/datasets/rasangabawanthi/mindbridge-fusion-artifacts/daisee_train_visual_features_rollfix.csv

daisee_validation_visual_features.csv
→ /kaggle/input/datasets/rasangabawanthi/mindbridge-fusion-artifacts/daisee_validation_visual_features.csv

daisee_test_visual_features.csv
→ /kaggle/input/datasets/rasangabawanthi/mindbridge-fusion-artifacts/daisee_test_visual_features.csv



In [6]:
# =========================================================
# STEP 5 — MERGE DISENGAGEMENT LABELS WITH VISUAL FEATURES
# =========================================================

import pandas as pd

# Existing visual feature files
train_feature_path = (
    "/kaggle/input/datasets/rasangabawanthi/"
    "mindbridge-fusion-artifacts/"
    "daisee_train_visual_features_rollfix.csv"
)

val_feature_path = (
    "/kaggle/input/datasets/rasangabawanthi/"
    "mindbridge-fusion-artifacts/"
    "daisee_validation_visual_features.csv"
)

test_feature_path = (
    "/kaggle/input/datasets/rasangabawanthi/"
    "mindbridge-fusion-artifacts/"
    "daisee_test_visual_features.csv"
)


# Load extracted visual features
train_visual = pd.read_csv(train_feature_path)
val_visual = pd.read_csv(val_feature_path)
test_visual = pd.read_csv(test_feature_path)


print("Before merge:")
print("Train:", train_visual.shape)
print("Validation:", val_visual.shape)
print("Test:", test_visual.shape)


# ---------------------------------------------------------
# Keep only labels needed for disengagement
# ---------------------------------------------------------

train_engagement_labels = train_labels[
    ["ClipID", "Engagement", "disengaged_label"]
].copy()

val_engagement_labels = val_labels[
    ["ClipID", "Engagement", "disengaged_label"]
].copy()

test_engagement_labels = test_labels[
    ["ClipID", "Engagement", "disengaged_label"]
].copy()


# ---------------------------------------------------------
# Merge by ClipID
# ---------------------------------------------------------

train_disengagement = train_visual.merge(
    train_engagement_labels,
    on="ClipID",
    how="left",
    validate="one_to_one"
)

val_disengagement = val_visual.merge(
    val_engagement_labels,
    on="ClipID",
    how="left",
    validate="one_to_one"
)

test_disengagement = test_visual.merge(
    test_engagement_labels,
    on="ClipID",
    how="left",
    validate="one_to_one"
)


# ---------------------------------------------------------
# Check results
# ---------------------------------------------------------

print("\nAfter merge:")
print("Train:", train_disengagement.shape)
print("Validation:", val_disengagement.shape)
print("Test:", test_disengagement.shape)


print("\nMissing disengagement labels:")
print(
    "Train:",
    train_disengagement["disengaged_label"].isna().sum()
)

print(
    "Validation:",
    val_disengagement["disengaged_label"].isna().sum()
)

print(
    "Test:",
    test_disengagement["disengaged_label"].isna().sum()
)


print("\nTRAIN class counts:")
print(
    train_disengagement[
        "disengaged_label"
    ].value_counts().sort_index()
)

print("\nVALIDATION class counts:")
print(
    val_disengagement[
        "disengaged_label"
    ].value_counts().sort_index()
)

print("\nTEST class counts:")
print(
    test_disengagement[
        "disengaged_label"
    ].value_counts().sort_index()
)


display(
    train_disengagement[
        [
            "ClipID",
            "Engagement",
            "disengaged_label",
            "mean_eye_openness",
            "mean_gaze_ratio",
            "mean_abs_pitch",
            "mean_abs_yaw",
            "face_visibility_ratio"
        ]
    ].head()
)

Before merge:
Train: (5358, 16)
Validation: (1429, 16)
Test: (1784, 16)

After merge:
Train: (5358, 18)
Validation: (1429, 18)
Test: (1784, 18)

Missing disengagement labels:
Train: 0
Validation: 0
Test: 0

TRAIN class counts:
disengaged_label
0    5111
1     247
Name: count, dtype: int64

VALIDATION class counts:
disengaged_label
0    1263
1     166
Name: count, dtype: int64

TEST class counts:
disengaged_label
0    1696
1      88
Name: count, dtype: int64


,ClipID,Engagement,disengaged_label,mean_eye_openness,mean_gaze_ratio,mean_abs_pitch,mean_abs_yaw,face_visibility_ratio
0,1100011002.avi,2,0,0.342717,0.477917,12.994088,6.043994,1.0
1,1100011003.avi,2,0,0.385494,0.480765,10.843975,8.098952,1.0
2,1100011004.avi,3,0,0.388826,0.488781,10.741835,9.218025,1.0
3,1100011005.avi,3,0,0.377390,0.482508,11.497001,4.637832,1.0
4,1100011006.avi,3,0,0.337944,0.472153,12.387394,6.481199,1.0


In [7]:
# =========================================================
# STEP 6 — CHECK VISUAL FEATURES BEFORE TRAINING
# =========================================================

visual_features = [
    "mean_eye_openness",
    "std_eye_openness",
    "mean_gaze_ratio",
    "std_gaze_ratio",
    "mean_pitch",
    "mean_yaw",
    "mean_roll",
    "mean_abs_pitch",
    "mean_abs_yaw",
    "mean_abs_roll",
    "face_visibility_ratio"
]

for name, df in [
    ("TRAIN", train_disengagement),
    ("VALIDATION", val_disengagement),
    ("TEST", test_disengagement)
]:

    print(f"\n===== {name} =====")

    print("Missing values:")
    print(df[visual_features].isna().sum())

    print(
        "\nTotal missing:",
        df[visual_features].isna().sum().sum()
    )

    print(
        "Infinite values:",
        (~df[visual_features]
          .applymap(lambda x: pd.notna(x) and abs(x) != float("inf"))
        ).sum().sum()
    )


===== TRAIN =====
Missing values:
mean_eye_openness        0
std_eye_openness         0
mean_gaze_ratio          0
std_gaze_ratio           0
mean_pitch               0
mean_yaw                 0
mean_roll                0
mean_abs_pitch           0
mean_abs_yaw             0
mean_abs_roll            0
face_visibility_ratio    0
dtype: int64

Total missing: 0
Infinite values: 0

===== VALIDATION =====
Missing values:
mean_eye_openness        0
std_eye_openness         0
mean_gaze_ratio          0
std_gaze_ratio           0
mean_pitch               0
mean_yaw                 0
mean_roll                0
mean_abs_pitch           0
mean_abs_yaw             0
mean_abs_roll            0
face_visibility_ratio    0
dtype: int64

Total missing: 0
Infinite values: 0

===== TEST =====
Missing values:
mean_eye_openness        0
std_eye_openness         0
mean_gaze_ratio          0
std_gaze_ratio           0
mean_pitch               0
mean_yaw                 0
mean_roll                0
mean_abs

/tmp/ipykernel_48/1458938826.py:38: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  .applymap(lambda x: pd.notna(x) and abs(x) != float("inf"))
/tmp/ipykernel_48/1458938826.py:38: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  .applymap(lambda x: pd.notna(x) and abs(x) != float("inf"))
/tmp/ipykernel_48/1458938826.py:38: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  .applymap(lambda x: pd.notna(x) and abs(x) != float("inf"))


In [8]:
# =========================================================
# STEP 7 — TRAIN VISUAL DISENGAGEMENT MODEL
# =========================================================

import numpy as np
from xgboost import XGBClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    classification_report,
    confusion_matrix
)


# ---------------------------------------------------------
# 1. Prepare train / validation data
# ---------------------------------------------------------

X_train = train_disengagement[visual_features]
y_train = train_disengagement["disengaged_label"].astype(int)

X_val = val_disengagement[visual_features]
y_val = val_disengagement["disengaged_label"].astype(int)


# ---------------------------------------------------------
# 2. Class imbalance weight
# ---------------------------------------------------------

negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = (
    negative_count / positive_count
)

print(
    "scale_pos_weight:",
    round(scale_pos_weight, 4)
)


# ---------------------------------------------------------
# 3. Build XGBoost model
# ---------------------------------------------------------

disengagement_visual_model = XGBClassifier(

    n_estimators=300,
    max_depth=4,
    learning_rate=0.03,

    subsample=0.8,
    colsample_bytree=0.8,

    min_child_weight=3,

    scale_pos_weight=scale_pos_weight,

    objective="binary:logistic",
    eval_metric="logloss",

    tree_method="hist",

    random_state=42,
    n_jobs=-1
)


# ---------------------------------------------------------
# 4. Train
# ---------------------------------------------------------

disengagement_visual_model.fit(
    X_train,
    y_train
)

print("\nVisual disengagement model trained ✅")


# ---------------------------------------------------------
# 5. Validation probabilities
# ---------------------------------------------------------

val_prob = disengagement_visual_model.predict_proba(
    X_val
)[:, 1]


# ---------------------------------------------------------
# 6. Probability-based metrics
# ---------------------------------------------------------

val_roc_auc = roc_auc_score(
    y_val,
    val_prob
)

val_pr_auc = average_precision_score(
    y_val,
    val_prob
)

print("\n===== VALIDATION =====")

print(
    "ROC-AUC:",
    round(val_roc_auc, 4)
)

print(
    "PR-AUC:",
    round(val_pr_auc, 4)
)


# ---------------------------------------------------------
# 7. Find best threshold using Macro F1
# ---------------------------------------------------------

thresholds = np.arange(
    0.05,
    0.96,
    0.01
)

results = []

for threshold in thresholds:

    pred = (
        val_prob >= threshold
    ).astype(int)

    macro_f1 = f1_score(
        y_val,
        pred,
        average="macro"
    )

    positive_f1 = f1_score(
        y_val,
        pred,
        pos_label=1
    )

    results.append(
        (
            threshold,
            macro_f1,
            positive_f1
        )
    )


best_threshold, best_macro_f1, best_positive_f1 = max(
    results,
    key=lambda x: x[1]
)


print(
    "\nBest threshold:",
    round(float(best_threshold), 2)
)

print(
    "Best Macro F1:",
    round(float(best_macro_f1), 4)
)

print(
    "Disengaged-class F1:",
    round(float(best_positive_f1), 4)
)


# ---------------------------------------------------------
# 8. Classification report at selected threshold
# ---------------------------------------------------------

val_pred = (
    val_prob >= best_threshold
).astype(int)

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        val_pred,
        digits=4
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_val,
        val_pred
    )
)

scale_pos_weight: 20.6923

Visual disengagement model trained ✅

===== VALIDATION =====
ROC-AUC: 0.6176
PR-AUC: 0.2045

Best threshold: 0.54
Best Macro F1: 0.5918
Disengaged-class F1: 0.2841

Classification Report:
              precision    recall  f1-score   support

           0     0.9067    0.8923    0.8994      1263
           1     0.2688    0.3012    0.2841       166

    accuracy                         0.8237      1429
   macro avg     0.5877    0.5968    0.5918      1429
weighted avg     0.8326    0.8237    0.8280      1429

Confusion Matrix:
[[1127  136]
 [ 116   50]]


In [9]:
# =========================================================
# STEP 8 — FINAL TEST EVALUATION
# DO NOT tune threshold using test data
# =========================================================

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    classification_report,
    confusion_matrix
)

X_test = test_disengagement[visual_features]

y_test = (
    test_disengagement["disengaged_label"]
    .astype(int)
)


# -----------------------------------------
# Test probabilities
# -----------------------------------------

test_prob = (
    disengagement_visual_model
    .predict_proba(X_test)[:, 1]
)


# -----------------------------------------
# Probability-based metrics
# -----------------------------------------

test_roc_auc = roc_auc_score(
    y_test,
    test_prob
)

test_pr_auc = average_precision_score(
    y_test,
    test_prob
)


# -----------------------------------------
# Use FIXED validation threshold
# -----------------------------------------

FINAL_THRESHOLD = float(best_threshold)

test_pred = (
    test_prob >= FINAL_THRESHOLD
).astype(int)


test_macro_f1 = f1_score(
    y_test,
    test_pred,
    average="macro"
)

test_positive_f1 = f1_score(
    y_test,
    test_pred,
    pos_label=1
)


print("===== FINAL TEST RESULTS =====")

print(
    "Fixed threshold:",
    round(FINAL_THRESHOLD, 2)
)

print(
    "ROC-AUC:",
    round(test_roc_auc, 4)
)

print(
    "PR-AUC:",
    round(test_pr_auc, 4)
)

print(
    "Macro F1:",
    round(test_macro_f1, 4)
)

print(
    "Disengaged-class F1:",
    round(test_positive_f1, 4)
)


print("\nClassification Report:")

print(
    classification_report(
        y_test,
        test_pred,
        digits=4
    )
)


print("Confusion Matrix:")

print(
    confusion_matrix(
        y_test,
        test_pred
    )
)

===== FINAL TEST RESULTS =====
Fixed threshold: 0.54
ROC-AUC: 0.6871
PR-AUC: 0.139
Macro F1: 0.5497
Disengaged-class F1: 0.1969

Classification Report:
              precision    recall  f1-score   support

           0     0.9664    0.8467    0.9026      1696
           1     0.1275    0.4318    0.1969        88

    accuracy                         0.8262      1784
   macro avg     0.5469    0.6393    0.5497      1784
weighted avg     0.9250    0.8262    0.8678      1784

Confusion Matrix:
[[1436  260]
 [  50   38]]


In [12]:
# =========================================================
# STEP 9 — SAVE VISUAL DISENGAGEMENT MODEL + METADATA
# =========================================================

import json

MODEL_PATH = (
    "/kaggle/working/"
    "mindbridge_disengagement_visual_model.json"
)

INFO_PATH = (
    "/kaggle/working/"
    "disengagement_visual_model_info.json"
)


# Save XGBoost model
disengagement_visual_model.save_model(
    MODEL_PATH
)


metadata = {

    "model_name":
        "MindBridge Visual Disengagement Baseline",

    "model_type":
        "XGBoost",

    "dataset":
        "DAiSEE",

    "target":
        "Low-engagement / disengagement proxy",

    "target_rule":
        "Engagement <= 1 -> disengaged_label = 1; Engagement >= 2 -> 0",

    "threshold":
        round(float(FINAL_THRESHOLD), 2),

    "features":
        visual_features,

    "train_class_distribution": {
        "engaged": 5111,
        "disengaged_proxy": 247
    },

    "validation_metrics": {
        "roc_auc": 0.6176,
        "pr_auc": 0.2045,
        "macro_f1": 0.5918,
        "disengaged_f1": 0.2841
    },

    "test_metrics": {
        "roc_auc": 0.6871,
        "pr_auc": 0.1390,
        "macro_f1": 0.5497,
        "disengaged_precision": 0.1275,
        "disengaged_recall": 0.4318,
        "disengaged_f1": 0.1969
    },

    "note":
        (
            "DAiSEE does not provide a direct disengagement label. "
            "Engagement levels 0 and 1 are used as a low-engagement "
            "proxy. This model is intended as a visual baseline and "
            "should be combined with behavioural evidence."
        )
}


with open(INFO_PATH, "w") as f:
    json.dump(
        metadata,
        f,
        indent=4
    )


print("Visual disengagement model saved ✅")
print(MODEL_PATH)

print("\nMetadata saved ✅")
print(INFO_PATH)

Visual disengagement model saved ✅
/kaggle/working/mindbridge_disengagement_visual_model.json

Metadata saved ✅
/kaggle/working/disengagement_visual_model_info.json


In [14]:
# =========================================================
# STEP 10 — CHECK ASSISTMENTS AFFECT / ENGAGEMENT COLUMNS
# =========================================================

import pandas as pd

ASSIST_PATH = (
    "/kaggle/input/datasets/rasangabawanthi/"
    "mindbridge-assistments-2012-2013/"
    "2012-2013-data-with-predictions-4-final.csv"
)

# Read only a few rows first
assist_preview = pd.read_csv(
    ASSIST_PATH,
    nrows=5
)

print("Total columns:", len(assist_preview.columns))

print("\n===== ALL COLUMNS =====")
for col in assist_preview.columns:
    print(col)


print("\n===== POSSIBLE AFFECT / ENGAGEMENT COLUMNS =====")

keywords = [
    "bored",
    "off",
    "engag",
    "concentr",
    "frustr",
    "confus",
    "confidence"
]

for col in assist_preview.columns:
    col_lower = col.lower()

    if any(
        keyword in col_lower
        for keyword in keywords
    ):
        print(col)

Total columns: 35

===== ALL COLUMNS =====
problem_log_id
skill
problem_id
user_id
assignment_id
assistment_id
start_time
end_time
problem_type
original
correct
bottom_hint
hint_count
actions
attempt_count
ms_first_response
tutor_mode
sequence_id
student_class_id
position
type
base_sequence_id
skill_id
teacher_id
school_id
overlap_time
template_id
answer_id
answer_text
first_action
problemlogid
Average_confidence(FRUSTRATED)
Average_confidence(CONFUSED)
Average_confidence(CONCENTRATING)
Average_confidence(BORED)

===== POSSIBLE AFFECT / ENGAGEMENT COLUMNS =====
Average_confidence(FRUSTRATED)
Average_confidence(CONFUSED)
Average_confidence(CONCENTRATING)
Average_confidence(BORED)


In [17]:
# =========================================================
# STEP 11 — INSPECT BEHAVIOURAL DISENGAGEMENT PROXIES
# =========================================================

import pandas as pd
import numpy as np

proxy_cols = [
    "Average_confidence(BORED)",
    "Average_confidence(CONCENTRATING)"
]

# Load only the two columns first to save memory
affect_df = pd.read_csv(
    ASSIST_PATH,
    usecols=proxy_cols
)

print("Shape:", affect_df.shape)


# ---------------------------------------------------------
# Missing values
# ---------------------------------------------------------

print("\n===== MISSING VALUES =====")

print(
    affect_df.isna().sum()
)


# ---------------------------------------------------------
# Distribution
# ---------------------------------------------------------

for col in proxy_cols:

    print(f"\n===== {col} =====")

    print(
        affect_df[col].describe(
            percentiles=[
                0.01,
                0.05,
                0.10,
                0.25,
                0.50,
                0.75,
                0.90,
                0.95,
                0.99
            ]
        )
    )


# ---------------------------------------------------------
# Candidate threshold = 0.50
# ---------------------------------------------------------

bored_positive = (
    affect_df[
        "Average_confidence(BORED)"
    ] >= 0.50
)

low_concentration = (
    affect_df[
        "Average_confidence(CONCENTRATING)"
    ] < 0.50
)


print("\n===== CANDIDATE PROXY COUNTS =====")

print(
    "BORED >= 0.50:",
    bored_positive.sum()
)

print(
    "BORED >= 0.50 percentage:",
    round(
        bored_positive.mean() * 100,
        2
    ),
    "%"
)


print(
    "\nCONCENTRATING < 0.50:",
    low_concentration.sum()
)

print(
    "CONCENTRATING < 0.50 percentage:",
    round(
        low_concentration.mean() * 100,
        2
    ),
    "%"
)


# ---------------------------------------------------------
# Combined stricter candidate
# ---------------------------------------------------------

combined_proxy = (
    bored_positive &
    low_concentration
)

print(
    "\nBORED >= 0.50 AND CONCENTRATING < 0.50:",
    combined_proxy.sum()
)

print(
    "Combined percentage:",
    round(
        combined_proxy.mean() * 100,
        2
    ),
    "%"
)


# ---------------------------------------------------------
# Relationship between boredom and concentration
# ---------------------------------------------------------

corr = affect_df[
    [
        "Average_confidence(BORED)",
        "Average_confidence(CONCENTRATING)"
    ]
].corr()

print("\n===== CORRELATION =====")

print(corr)

Shape: (6123270, 2)

===== MISSING VALUES =====
Average_confidence(CONCENTRATING)    0
Average_confidence(BORED)            0
dtype: int64

===== Average_confidence(BORED) =====
count    6.123270e+06
mean     2.567723e-01
std      2.862460e-01
min      0.000000e+00
1%       0.000000e+00
5%       0.000000e+00
10%      0.000000e+00
25%      0.000000e+00
50%      2.214840e-01
75%      4.429680e-01
90%      7.626890e-01
95%      8.259197e-01
99%      9.523810e-01
max      1.000000e+00
Name: Average_confidence(BORED), dtype: float64

===== Average_confidence(CONCENTRATING) =====
count    6.123270e+06
mean     6.823843e-01
std      1.713734e-01
min      1.707320e-01
1%       2.291624e-01
5%       3.365290e-01
10%      3.365290e-01
25%      7.669250e-01
50%      7.669250e-01
75%      7.669250e-01
90%      7.669250e-01
95%      7.669250e-01
99%      7.669250e-01
max      7.669250e-01
Name: Average_confidence(CONCENTRATING), dtype: float64

===== CANDIDATE PROXY COUNTS =====
BORED >= 0.50: 9540

In [18]:
# =========================================================
# STEP 12 — BUILD BEHAVIOURAL DISENGAGEMENT DATASET
# =========================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split


behavior_cols = [
    "user_id",
    "correct",
    "bottom_hint",
    "hint_count",
    "attempt_count",
    "ms_first_response",
    "overlap_time",
    "first_action",
    "position",
    "original",
    "Average_confidence(BORED)"
]


behavior_disengagement_df = pd.read_csv(
    ASSIST_PATH,
    usecols=behavior_cols
)

print(
    "Loaded shape:",
    behavior_disengagement_df.shape
)


# ---------------------------------------------------------
# 1. Create target
# ---------------------------------------------------------

behavior_disengagement_df[
    "disengaged_label"
] = (
    behavior_disengagement_df[
        "Average_confidence(BORED)"
    ] >= 0.50
).astype(int)


print("\nTarget counts:")

print(
    behavior_disengagement_df[
        "disengaged_label"
    ].value_counts().sort_index()
)


print("\nTarget percentages:")

print(
    (
        behavior_disengagement_df[
            "disengaged_label"
        ]
        .value_counts(normalize=True)
        .sort_index() * 100
    ).round(2)
)


# ---------------------------------------------------------
# 2. Missing indicator
# ---------------------------------------------------------

behavior_disengagement_df[
    "bottom_hint_missing"
] = (
    behavior_disengagement_df[
        "bottom_hint"
    ].isna()
).astype(int)


# Fill bottom_hint using TRAIN median later.
# For learner split we don't need to fill it yet.


# ---------------------------------------------------------
# 3. Learner-wise split
# ---------------------------------------------------------

users = (
    behavior_disengagement_df[
        "user_id"
    ]
    .drop_duplicates()
    .to_numpy()
)


train_users, temp_users = train_test_split(
    users,
    test_size=0.30,
    random_state=42
)


val_users, test_users = train_test_split(
    temp_users,
    test_size=0.50,
    random_state=42
)


train_mask = (
    behavior_disengagement_df[
        "user_id"
    ].isin(train_users)
)

val_mask = (
    behavior_disengagement_df[
        "user_id"
    ].isin(val_users)
)

test_mask = (
    behavior_disengagement_df[
        "user_id"
    ].isin(test_users)
)


print("\n===== LEARNER SPLIT =====")

print(
    "Train learners:",
    len(train_users)
)

print(
    "Validation learners:",
    len(val_users)
)

print(
    "Test learners:",
    len(test_users)
)


print("\n===== ROW COUNTS =====")

print(
    "Train rows:",
    train_mask.sum()
)

print(
    "Validation rows:",
    val_mask.sum()
)

print(
    "Test rows:",
    test_mask.sum()
)


# ---------------------------------------------------------
# 4. Check positive rate per split
# ---------------------------------------------------------

for name, mask in [
    ("TRAIN", train_mask),
    ("VALIDATION", val_mask),
    ("TEST", test_mask)
]:

    subset = behavior_disengagement_df.loc[
        mask,
        "disengaged_label"
    ]

    print(f"\n===== {name} =====")

    print(
        subset.value_counts().sort_index()
    )

    print(
        "Positive rate:",
        round(
            subset.mean() * 100,
            2
        ),
        "%"
    )

Loaded shape: (6123270, 11)

Target counts:
disengaged_label
0    5169269
1     954001
Name: count, dtype: int64

Target percentages:
disengaged_label
0    84.42
1    15.58
Name: proportion, dtype: float64

===== LEARNER SPLIT =====
Train learners: 32671
Validation learners: 7001
Test learners: 7002

===== ROW COUNTS =====
Train rows: 4263157
Validation rows: 925020
Test rows: 935093

===== TRAIN =====
disengaged_label
0    3599074
1     664083
Name: count, dtype: int64
Positive rate: 15.58 %

===== VALIDATION =====
disengaged_label
0    781495
1    143525
Name: count, dtype: int64
Positive rate: 15.52 %

===== TEST =====
disengaged_label
0    788700
1    146393
Name: count, dtype: int64
Positive rate: 15.66 %


In [19]:
# =========================================================
# STEP 13 — TRAIN BEHAVIOURAL DISENGAGEMENT MODEL
# =========================================================

import numpy as np
from xgboost import XGBClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    classification_report,
    confusion_matrix
)


# ---------------------------------------------------------
# 1. Behaviour feature set
# ---------------------------------------------------------

behavior_features = [
    "correct",
    "bottom_hint",
    "hint_count",
    "attempt_count",
    "ms_first_response",
    "overlap_time",
    "first_action",
    "position",
    "original",
    "bottom_hint_missing"
]


# ---------------------------------------------------------
# 2. TRAIN-ONLY preprocessing
# ---------------------------------------------------------

bottom_hint_median = (
    behavior_disengagement_df.loc[
        train_mask,
        "bottom_hint"
    ].median()
)

print(
    "Train bottom_hint median:",
    bottom_hint_median
)


def prepare_behavior_X(mask):

    X = behavior_disengagement_df.loc[
        mask,
        behavior_features
    ].copy()

    # Use TRAIN median only
    X["bottom_hint"] = (
        X["bottom_hint"]
        .fillna(bottom_hint_median)
    )

    # Reduce memory usage
    X = X.astype(np.float32)

    return X


X_train_beh = prepare_behavior_X(
    train_mask
)

X_val_beh = prepare_behavior_X(
    val_mask
)


y_train_beh = (
    behavior_disengagement_df.loc[
        train_mask,
        "disengaged_label"
    ]
    .astype(np.uint8)
)

y_val_beh = (
    behavior_disengagement_df.loc[
        val_mask,
        "disengaged_label"
    ]
    .astype(np.uint8)
)


print("\nTrain X shape:", X_train_beh.shape)
print("Validation X shape:", X_val_beh.shape)


# ---------------------------------------------------------
# 3. Class weight
# ---------------------------------------------------------

negative_count = (
    y_train_beh == 0
).sum()

positive_count = (
    y_train_beh == 1
).sum()

behavior_scale_pos_weight = (
    negative_count /
    positive_count
)

print(
    "\nscale_pos_weight:",
    round(
        float(behavior_scale_pos_weight),
        4
    )
)


# ---------------------------------------------------------
# 4. Build XGBoost model
# ---------------------------------------------------------

disengagement_behavior_model = XGBClassifier(

    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,

    subsample=0.8,
    colsample_bytree=0.8,

    min_child_weight=3,

    scale_pos_weight=
        behavior_scale_pos_weight,

    objective="binary:logistic",
    eval_metric="logloss",

    tree_method="hist",

    random_state=42,
    n_jobs=-1
)


# ---------------------------------------------------------
# 5. Train
# ---------------------------------------------------------

disengagement_behavior_model.fit(
    X_train_beh,
    y_train_beh
)

print(
    "\nBehaviour disengagement model trained ✅"
)


# ---------------------------------------------------------
# 6. Validation probabilities
# ---------------------------------------------------------

val_behavior_prob = (
    disengagement_behavior_model
    .predict_proba(X_val_beh)[:, 1]
)


# ---------------------------------------------------------
# 7. Validation probability metrics
# ---------------------------------------------------------

val_behavior_roc = roc_auc_score(
    y_val_beh,
    val_behavior_prob
)

val_behavior_pr = average_precision_score(
    y_val_beh,
    val_behavior_prob
)


print("\n===== VALIDATION =====")

print(
    "ROC-AUC:",
    round(val_behavior_roc, 4)
)

print(
    "PR-AUC:",
    round(val_behavior_pr, 4)
)


# ---------------------------------------------------------
# 8. Find best validation threshold
#    using Macro F1
# ---------------------------------------------------------

thresholds = np.arange(
    0.05,
    0.96,
    0.01
)

threshold_results = []

for threshold in thresholds:

    pred = (
        val_behavior_prob >= threshold
    ).astype(int)

    macro_f1 = f1_score(
        y_val_beh,
        pred,
        average="macro"
    )

    disengaged_f1 = f1_score(
        y_val_beh,
        pred,
        pos_label=1
    )

    threshold_results.append(
        (
            threshold,
            macro_f1,
            disengaged_f1
        )
    )


(
    best_behavior_threshold,
    best_behavior_macro_f1,
    best_behavior_positive_f1
) = max(
    threshold_results,
    key=lambda x: x[1]
)


print(
    "\nBest threshold:",
    round(
        float(best_behavior_threshold),
        2
    )
)

print(
    "Best Macro F1:",
    round(
        float(best_behavior_macro_f1),
        4
    )
)

print(
    "Disengaged-class F1:",
    round(
        float(best_behavior_positive_f1),
        4
    )
)


# ---------------------------------------------------------
# 9. Validation report
# ---------------------------------------------------------

val_behavior_pred = (
    val_behavior_prob >=
    best_behavior_threshold
).astype(int)


print(
    "\nClassification Report:"
)

print(
    classification_report(
        y_val_beh,
        val_behavior_pred,
        digits=4
    )
)


print(
    "Confusion Matrix:"
)

print(
    confusion_matrix(
        y_val_beh,
        val_behavior_pred
    )
)

Train bottom_hint median: 0.0

Train X shape: (4263157, 10)
Validation X shape: (925020, 10)

scale_pos_weight: 5.4196

Behaviour disengagement model trained ✅

===== VALIDATION =====
ROC-AUC: 0.529
PR-AUC: 0.1679

Best threshold: 0.53
Best Macro F1: 0.5126
Disengaged-class F1: 0.1739

Classification Report:
              precision    recall  f1-score   support

           0     0.8487    0.8541    0.8514    781495
           1     0.1770    0.1709    0.1739    143525

    accuracy                         0.7481    925020
   macro avg     0.5128    0.5125    0.5126    925020
weighted avg     0.7445    0.7481    0.7463    925020

Confusion Matrix:
[[667454 114041]
 [119002  24523]]


In [20]:
behavior_disengagement_v1_results = {
    "model": "XGBoost Behaviour Disengagement Baseline V1",
    "target_rule": "Average_confidence(BORED) >= 0.50",
    "validation_roc_auc": 0.5290,
    "validation_pr_auc": 0.1679,
    "validation_macro_f1": 0.5126,
    "validation_disengaged_f1": 0.1739,
    "threshold": 0.53,
    "status": "weak_static_baseline"
}

print("Behaviour Disengagement V1 baseline recorded ✅")

Behaviour Disengagement V1 baseline recorded ✅


In [21]:
# =========================================================
# STEP 14A — ADD START TIME FOR TEMPORAL FEATURES
# =========================================================

import pandas as pd

start_time_df = pd.read_csv(
    ASSIST_PATH,
    usecols=["start_time"]
)

print("Start-time rows:", len(start_time_df))
print("Behaviour rows:", len(behavior_disengagement_df))

assert len(start_time_df) == len(behavior_disengagement_df), \
    "Row count mismatch!"

behavior_disengagement_df["start_time"] = pd.to_datetime(
    start_time_df["start_time"],
    errors="coerce"
)

del start_time_df


print(
    "\nMissing parsed start_time:",
    behavior_disengagement_df[
        "start_time"
    ].isna().sum()
)

print("\nExample:")
display(
    behavior_disengagement_df[
        [
            "user_id",
            "start_time",
            "correct",
            "ms_first_response",
            "attempt_count",
            "hint_count",
            "disengaged_label"
        ]
    ].head()
)

Start-time rows: 6123270
Behaviour rows: 6123270

Missing parsed start_time: 73588

Example:


,user_id,start_time,correct,ms_first_response,attempt_count,hint_count,disengaged_label
0,61394,2012-09-28 15:11:27,1.0,9852,1,0,0
1,61394,2012-10-09 11:01:52,1.0,21175,1,0,0
2,61394,2013-03-07 10:53:20,0.0,8645,1,0,0
3,61394,2013-08-20 19:54:56,1.0,25728,1,0,1
4,76592,2012-09-10 17:20:10,1.0,286578,1,0,0


In [22]:
# =========================================================
# STEP 14B — INSPECT UNPARSED START TIMES
# =========================================================

raw_start_time = pd.read_csv(
    ASSIST_PATH,
    usecols=["start_time"],
    dtype={"start_time": "string"}
)["start_time"]

parsed_start_time = pd.to_datetime(
    raw_start_time,
    errors="coerce"
)

bad_mask = parsed_start_time.isna()

print("Total rows:", len(raw_start_time))
print("Unparsed rows:", bad_mask.sum())

print(
    "\nRaw missing/blank values:",
    raw_start_time[bad_mask].isna().sum()
)

print("\n===== SAMPLE UNPARSED VALUES =====")

print(
    raw_start_time[
        bad_mask
    ]
    .dropna()
    .drop_duplicates()
    .head(30)
    .to_string(index=False)
)

Total rows: 6123270
Unparsed rows: 73588

Raw missing/blank values: 0

===== SAMPLE UNPARSED VALUES =====
2012-09-03 14:18:36.278543
 2012-09-04 16:28:52.97775
 2012-09-02 10:22:46.90231
2012-10-12 11:07:08.717064
2012-09-04 08:57:48.071466
2012-09-03 22:22:02.108292
2013-02-14 11:48:21.150174
2013-03-13 11:42:56.174801
2013-03-13 11:59:09.735549
2013-02-26 20:32:31.102635
2013-02-07 11:48:01.469551
2013-02-07 10:23:49.986323
2013-02-07 10:09:50.559826
2013-02-27 11:45:33.822721
2012-10-30 23:17:53.658617
2013-05-28 20:49:26.307073
2012-09-24 10:58:29.591698
2012-09-09 16:17:55.608988
2012-09-01 00:00:38.696652
 2012-09-30 13:55:54.10627
2012-09-09 21:24:42.502453
 2012-09-09 12:19:48.56071
2012-09-02 15:12:52.678413
2012-09-01 00:24:01.343715
2012-09-02 03:17:33.645167
2012-09-07 10:28:41.012139
2013-05-30 08:07:08.046557
2012-10-18 09:18:03.909399
2012-10-26 09:03:25.437287
2012-12-11 21:00:35.225728


In [23]:
# =========================================================
# STEP 14C — FIX MIXED START_TIME FORMATS
# =========================================================

raw_start_time = pd.read_csv(
    ASSIST_PATH,
    usecols=["start_time"],
    dtype={"start_time": "string"}
)["start_time"]

# Remove leading/trailing spaces
clean_start_time = raw_start_time.str.strip()

# Parse mixed formats safely
fixed_start_time = pd.to_datetime(
    clean_start_time,
    format="mixed",
    errors="coerce"
)

print(
    "Remaining unparsed rows:",
    fixed_start_time.isna().sum()
)

print("\nFirst 5 parsed values:")
print(fixed_start_time.head())


# Replace old parsed column
behavior_disengagement_df["start_time"] = fixed_start_time

print(
    "\nStored missing start_time:",
    behavior_disengagement_df[
        "start_time"
    ].isna().sum()
)

Remaining unparsed rows: 0

First 5 parsed values:
0   2012-09-28 15:11:27
1   2012-10-09 11:01:52
2   2013-03-07 10:53:20
3   2013-08-20 19:54:56
4   2012-09-10 17:20:10
Name: start_time, dtype: datetime64[ns]

Stored missing start_time: 0


In [24]:
# =========================================================
# STEP 14D — LEARNER-SPECIFIC TEMPORAL FEATURES
# Previous 20 interactions ONLY
# =========================================================

import numpy as np
import pandas as pd


# ---------------------------------------------------------
# 1. Preserve original order for reference
# ---------------------------------------------------------

behavior_disengagement_df["original_row_id"] = np.arange(
    len(behavior_disengagement_df)
)


# ---------------------------------------------------------
# 2. Sort chronologically inside each learner
# ---------------------------------------------------------

behavior_disengagement_df = (
    behavior_disengagement_df
    .sort_values(
        [
            "user_id",
            "start_time",
            "original_row_id"
        ],
        kind="mergesort"
    )
    .reset_index(drop=True)
)

print("Sorted by learner + time ✅")


# ---------------------------------------------------------
# 3. Clean response time for temporal calculations
#    Negative response times are invalid
# ---------------------------------------------------------

valid_rt = (
    behavior_disengagement_df[
        "ms_first_response"
    ].astype(float)
)

valid_rt = valid_rt.mask(
    valid_rt < 0,
    np.nan
)

behavior_disengagement_df[
    "log_response_time"
] = np.log1p(valid_rt)


# ---------------------------------------------------------
# 4. Number of PREVIOUS interactions
# ---------------------------------------------------------

behavior_disengagement_df[
    "previous_interactions"
] = (
    behavior_disengagement_df
    .groupby(
        "user_id",
        sort=False
    )
    .cumcount()
)


# ---------------------------------------------------------
# 5. Helper:
#    shift(1) prevents current/future leakage
# ---------------------------------------------------------

def previous_rolling_mean(column):

    return (
        behavior_disengagement_df
        .groupby(
            "user_id",
            sort=False
        )[column]
        .transform(
            lambda s:
                s.shift(1)
                 .rolling(
                     window=20,
                     min_periods=5
                 )
                 .mean()
        )
    )


print("Creating previous-20 interaction features...")


# ---------------------------------------------------------
# 6. Recent learner baselines
# ---------------------------------------------------------

behavior_disengagement_df[
    "recent_log_rt"
] = previous_rolling_mean(
    "log_response_time"
)

behavior_disengagement_df[
    "recent_accuracy"
] = previous_rolling_mean(
    "correct"
)

behavior_disengagement_df[
    "recent_attempts"
] = previous_rolling_mean(
    "attempt_count"
)

behavior_disengagement_df[
    "recent_hints"
] = previous_rolling_mean(
    "hint_count"
)


# ---------------------------------------------------------
# 7. Current behaviour vs learner's recent baseline
# ---------------------------------------------------------

behavior_disengagement_df[
    "rt_deviation"
] = (
    behavior_disengagement_df[
        "log_response_time"
    ]
    -
    behavior_disengagement_df[
        "recent_log_rt"
    ]
)

behavior_disengagement_df[
    "attempt_deviation"
] = (
    behavior_disengagement_df[
        "attempt_count"
    ]
    -
    behavior_disengagement_df[
        "recent_attempts"
    ]
)

behavior_disengagement_df[
    "hint_deviation"
] = (
    behavior_disengagement_df[
        "hint_count"
    ]
    -
    behavior_disengagement_df[
        "recent_hints"
    ]
)

behavior_disengagement_df[
    "accuracy_deviation"
] = (
    behavior_disengagement_df[
        "correct"
    ]
    -
    behavior_disengagement_df[
        "recent_accuracy"
    ]
)


# ---------------------------------------------------------
# 8. How much history do we have?
# ---------------------------------------------------------

behavior_disengagement_df[
    "history_confidence"
] = np.clip(
    behavior_disengagement_df[
        "previous_interactions"
    ] / 20.0,
    0,
    1
).astype(np.float32)


behavior_disengagement_df[
    "baseline_available"
] = (
    behavior_disengagement_df[
        "previous_interactions"
    ] >= 5
).astype(np.uint8)


# ---------------------------------------------------------
# 9. Check coverage
# ---------------------------------------------------------

baseline_rows = (
    behavior_disengagement_df[
        "baseline_available"
    ].sum()
)

coverage = (
    baseline_rows /
    len(behavior_disengagement_df)
) * 100


print("\n===== TEMPORAL FEATURE SUMMARY =====")

print(
    "Total rows:",
    len(behavior_disengagement_df)
)

print(
    "Rows with >=5 previous interactions:",
    baseline_rows
)

print(
    "Baseline coverage:",
    round(coverage, 2),
    "%"
)


temporal_features = [
    "recent_log_rt",
    "recent_accuracy",
    "recent_attempts",
    "recent_hints",
    "rt_deviation",
    "attempt_deviation",
    "hint_deviation",
    "accuracy_deviation",
    "history_confidence"
]


print("\nMissing values:")

print(
    behavior_disengagement_df[
        temporal_features
    ].isna().sum()
)


print("\nExample learner history:")

example_user = (
    behavior_disengagement_df[
        "user_id"
    ].value_counts().index[0]
)

display(
    behavior_disengagement_df[
        behavior_disengagement_df[
            "user_id"
        ] == example_user
    ][
        [
            "user_id",
            "start_time",
            "previous_interactions",
            "correct",
            "recent_accuracy",
            "ms_first_response",
            "recent_log_rt",
            "rt_deviation",
            "history_confidence",
            "disengaged_label"
        ]
    ].head(10)
)

Sorted by learner + time ✅
Creating previous-20 interaction features...

===== TEMPORAL FEATURE SUMMARY =====
Total rows: 6123270
Rows with >=5 previous interactions: 5897943
Baseline coverage: 96.32 %

Missing values:
recent_log_rt         225327
recent_accuracy       225327
recent_attempts       225327
recent_hints          225327
rt_deviation          225340
attempt_deviation     225327
hint_deviation        225327
accuracy_deviation    225327
history_confidence         0
dtype: int64

Example learner history:


,user_id,start_time,previous_interactions,correct,recent_accuracy,ms_first_response,recent_log_rt,rt_deviation,history_confidence,disengaged_label
5818408,221054,2013-06-24 09:25:40,0,1.0,NaN,2098,NaN,NaN,0.00,0
5818409,221054,2013-06-24 09:25:42,1,1.0,NaN,2319,NaN,NaN,0.05,0
5818410,221054,2013-06-24 09:25:44,2,1.0,NaN,3638,NaN,NaN,0.10,0
5818411,221054,2013-06-24 09:25:47,3,1.0,NaN,6464,NaN,NaN,0.15,0
5818412,221054,2013-06-24 09:25:53,4,1.0,NaN,3927,NaN,NaN,0.20,0
5818413,221054,2013-06-24 09:25:57,5,1.0,1.0,1157,8.129609,-1.075160,0.25,0
5818414,221054,2013-06-24 09:25:58,6,1.0,1.0,1040,7.950416,-1.002479,0.30,0
5818415,221054,2013-06-24 09:25:59,7,1.0,1.0,8710,7.807205,1.265137,0.35,0
5818416,221054,2013-06-24 09:26:08,8,1.0,1.0,2407,7.965347,-0.178795,0.40,0
5818417,221054,2013-06-24 09:26:10,9,1.0,1.0,5590,7.945481,0.683433,0.45,0


In [25]:
# =========================================================
# STEP 14E — REBUILD LEARNER SPLITS AFTER SORTING
# =========================================================

train_mask_v2 = (
    behavior_disengagement_df["user_id"]
    .isin(train_users)
)

val_mask_v2 = (
    behavior_disengagement_df["user_id"]
    .isin(val_users)
)

test_mask_v2 = (
    behavior_disengagement_df["user_id"]
    .isin(test_users)
)


print("===== ROW COUNTS AFTER SORTING =====")

print("Train:", train_mask_v2.sum())
print("Validation:", val_mask_v2.sum())
print("Test:", test_mask_v2.sum())


# Safety check: learner overlap must remain zero
train_set = set(
    behavior_disengagement_df.loc[
        train_mask_v2,
        "user_id"
    ].unique()
)

val_set = set(
    behavior_disengagement_df.loc[
        val_mask_v2,
        "user_id"
    ].unique()
)

test_set = set(
    behavior_disengagement_df.loc[
        test_mask_v2,
        "user_id"
    ].unique()
)


print("\n===== LEARNER OVERLAP =====")

print(
    "Train ∩ Validation:",
    len(train_set & val_set)
)

print(
    "Train ∩ Test:",
    len(train_set & test_set)
)

print(
    "Validation ∩ Test:",
    len(val_set & test_set)
)

===== ROW COUNTS AFTER SORTING =====
Train: 4263157
Validation: 925020
Test: 935093

===== LEARNER OVERLAP =====
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [26]:
# =========================================================
# STEP 15 — BEHAVIOURAL DISENGAGEMENT V2
# Static + learner-specific temporal features
# =========================================================

import gc
import numpy as np

from xgboost import XGBClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    classification_report,
    confusion_matrix
)


# ---------------------------------------------------------
# 0. Free old feature matrices if still in memory
# ---------------------------------------------------------

for var_name in ["X_train_beh", "X_val_beh"]:
    if var_name in globals():
        del globals()[var_name]

gc.collect()


# ---------------------------------------------------------
# 1. V2 features
# ---------------------------------------------------------

behavior_features_v2 = [

    # ----- Original/static behaviour -----
    "correct",
    "bottom_hint",
    "hint_count",
    "attempt_count",
    "ms_first_response",
    "overlap_time",
    "first_action",
    "position",
    "original",
    "bottom_hint_missing",

    # ----- Previous-20 learner history -----
    "recent_log_rt",
    "recent_accuracy",
    "recent_attempts",
    "recent_hints",

    # ----- Current vs learner baseline -----
    "rt_deviation",
    "attempt_deviation",
    "hint_deviation",
    "accuracy_deviation",

    # ----- Reliability of learner history -----
    "history_confidence",
    "baseline_available"
]


print("V2 feature count:", len(behavior_features_v2))


# ---------------------------------------------------------
# 2. Use TRAIN-ONLY median for bottom_hint
# ---------------------------------------------------------

bottom_hint_median_v2 = (
    behavior_disengagement_df.loc[
        train_mask_v2,
        "bottom_hint"
    ].median()
)

print(
    "Train bottom_hint median:",
    bottom_hint_median_v2
)


def prepare_behavior_v2(mask):

    X = behavior_disengagement_df.loc[
        mask,
        behavior_features_v2
    ].copy()

    # Same preprocessing as V1
    X["bottom_hint"] = (
        X["bottom_hint"]
        .fillna(bottom_hint_median_v2)
    )

    # Keep temporal NaNs.
    # XGBoost handles missing values natively.
    X = X.astype(np.float32)

    return X


X_train_v2 = prepare_behavior_v2(
    train_mask_v2
)

X_val_v2 = prepare_behavior_v2(
    val_mask_v2
)


y_train_v2 = (
    behavior_disengagement_df.loc[
        train_mask_v2,
        "disengaged_label"
    ]
    .astype(np.uint8)
)

y_val_v2 = (
    behavior_disengagement_df.loc[
        val_mask_v2,
        "disengaged_label"
    ]
    .astype(np.uint8)
)


print("\nTrain shape:", X_train_v2.shape)
print("Validation shape:", X_val_v2.shape)

print(
    "Train temporal NaNs:",
    int(
        X_train_v2[
            [
                "recent_log_rt",
                "recent_accuracy",
                "recent_attempts",
                "recent_hints",
                "rt_deviation",
                "attempt_deviation",
                "hint_deviation",
                "accuracy_deviation"
            ]
        ].isna().sum().sum()
    )
)


# ---------------------------------------------------------
# 3. Class weight
# ---------------------------------------------------------

negative_v2 = (
    y_train_v2 == 0
).sum()

positive_v2 = (
    y_train_v2 == 1
).sum()

scale_pos_weight_v2 = (
    negative_v2 / positive_v2
)

print(
    "\nscale_pos_weight:",
    round(
        float(scale_pos_weight_v2),
        4
    )
)


# ---------------------------------------------------------
# 4. Same model settings as V1
# ---------------------------------------------------------

disengagement_behavior_model_v2 = XGBClassifier(

    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,

    subsample=0.8,
    colsample_bytree=0.8,

    min_child_weight=3,

    scale_pos_weight=scale_pos_weight_v2,

    objective="binary:logistic",
    eval_metric="logloss",

    tree_method="hist",

    random_state=42,
    n_jobs=-1
)


# ---------------------------------------------------------
# 5. Train
# ---------------------------------------------------------

disengagement_behavior_model_v2.fit(
    X_train_v2,
    y_train_v2
)

print(
    "\nBehaviour Disengagement V2 trained ✅"
)


# ---------------------------------------------------------
# 6. Validation probabilities
# ---------------------------------------------------------

val_prob_v2 = (
    disengagement_behavior_model_v2
    .predict_proba(X_val_v2)[:, 1]
)


# ---------------------------------------------------------
# 7. Probability metrics
# ---------------------------------------------------------

roc_v2 = roc_auc_score(
    y_val_v2,
    val_prob_v2
)

pr_v2 = average_precision_score(
    y_val_v2,
    val_prob_v2
)


print("\n===== V2 VALIDATION =====")

print(
    "ROC-AUC:",
    round(roc_v2, 4)
)

print(
    "PR-AUC:",
    round(pr_v2, 4)
)


# ---------------------------------------------------------
# 8. Find best threshold using Macro F1
# ---------------------------------------------------------

threshold_results_v2 = []

for threshold in np.arange(
    0.05,
    0.96,
    0.01
):

    pred = (
        val_prob_v2 >= threshold
    ).astype(np.uint8)

    macro_f1 = f1_score(
        y_val_v2,
        pred,
        average="macro"
    )

    disengaged_f1 = f1_score(
        y_val_v2,
        pred,
        pos_label=1
    )

    threshold_results_v2.append(
        (
            threshold,
            macro_f1,
            disengaged_f1
        )
    )


(
    best_threshold_v2,
    best_macro_f1_v2,
    best_positive_f1_v2
) = max(
    threshold_results_v2,
    key=lambda x: x[1]
)


print(
    "\nBest threshold:",
    round(
        float(best_threshold_v2),
        2
    )
)

print(
    "Best Macro F1:",
    round(
        float(best_macro_f1_v2),
        4
    )
)

print(
    "Disengaged-class F1:",
    round(
        float(best_positive_f1_v2),
        4
    )
)


# ---------------------------------------------------------
# 9. Validation report
# ---------------------------------------------------------

val_pred_v2 = (
    val_prob_v2 >=
    best_threshold_v2
).astype(np.uint8)


print("\nClassification Report:")

print(
    classification_report(
        y_val_v2,
        val_pred_v2,
        digits=4
    )
)

print("Confusion Matrix:")

print(
    confusion_matrix(
        y_val_v2,
        val_pred_v2
    )
)


# ---------------------------------------------------------
# 10. Direct comparison with V1
# ---------------------------------------------------------

print("\n===== V1 vs V2 =====")

print("V1 ROC-AUC :", 0.5290)
print("V2 ROC-AUC :", round(roc_v2, 4))

print()

print("V1 PR-AUC  :", 0.1679)
print("V2 PR-AUC  :", round(pr_v2, 4))

print()

print("V1 Macro F1:", 0.5126)
print(
    "V2 Macro F1:",
    round(
        float(best_macro_f1_v2),
        4
    )
)

V2 feature count: 20
Train bottom_hint median: 0.0

Train shape: (4263157, 20)
Validation shape: (925020, 20)
Train temporal NaNs: 1261549

scale_pos_weight: 5.4196

Behaviour Disengagement V2 trained ✅

===== V2 VALIDATION =====
ROC-AUC: 0.536
PR-AUC: 0.1712

Best threshold: 0.53
Best Macro F1: 0.5153
Disengaged-class F1: 0.183

Classification Report:
              precision    recall  f1-score   support

           0     0.8497    0.8455    0.8476    781495
           1     0.1807    0.1855    0.1830    143525

    accuracy                         0.7431    925020
   macro avg     0.5152    0.5155    0.5153    925020
weighted avg     0.7459    0.7431    0.7445    925020

Confusion Matrix:
[[660780 120715]
 [116904  26621]]

===== V1 vs V2 =====
V1 ROC-AUC : 0.529
V2 ROC-AUC : 0.536

V1 PR-AUC  : 0.1679
V2 PR-AUC  : 0.1712

V1 Macro F1: 0.5126
V2 Macro F1: 0.5153


In [27]:
# =========================================================
# STEP 16 — FINAL TEST: BEHAVIOUR DISENGAGEMENT V2
# Threshold fixed from validation
# =========================================================

import gc
import numpy as np

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    classification_report,
    confusion_matrix
)


# ---------------------------------------------------------
# 1. Free large training matrices if needed
# ---------------------------------------------------------

if "X_train_v2" in globals():
    del X_train_v2

gc.collect()


# ---------------------------------------------------------
# 2. Prepare TEST features
# ---------------------------------------------------------

X_test_v2 = prepare_behavior_v2(
    test_mask_v2
)

y_test_v2 = (
    behavior_disengagement_df.loc[
        test_mask_v2,
        "disengaged_label"
    ]
    .astype(np.uint8)
)


print(
    "Test shape:",
    X_test_v2.shape
)


# ---------------------------------------------------------
# 3. Predict probabilities
# ---------------------------------------------------------

test_prob_v2 = (
    disengagement_behavior_model_v2
    .predict_proba(X_test_v2)[:, 1]
)


# ---------------------------------------------------------
# 4. Fixed validation threshold
# ---------------------------------------------------------

FINAL_BEHAVIOR_DISENGAGEMENT_THRESHOLD = float(
    best_threshold_v2
)

test_pred_v2 = (
    test_prob_v2 >=
    FINAL_BEHAVIOR_DISENGAGEMENT_THRESHOLD
).astype(np.uint8)


# ---------------------------------------------------------
# 5. Final metrics
# ---------------------------------------------------------

test_roc_v2 = roc_auc_score(
    y_test_v2,
    test_prob_v2
)

test_pr_v2 = average_precision_score(
    y_test_v2,
    test_prob_v2
)

test_macro_f1_v2 = f1_score(
    y_test_v2,
    test_pred_v2,
    average="macro"
)

test_positive_f1_v2 = f1_score(
    y_test_v2,
    test_pred_v2,
    pos_label=1
)


print("\n===== FINAL TEST RESULTS — V2 =====")

print(
    "Fixed threshold:",
    round(
        FINAL_BEHAVIOR_DISENGAGEMENT_THRESHOLD,
        2
    )
)

print(
    "ROC-AUC:",
    round(test_roc_v2, 4)
)

print(
    "PR-AUC:",
    round(test_pr_v2, 4)
)

print(
    "Macro F1:",
    round(test_macro_f1_v2, 4)
)

print(
    "Disengaged-class F1:",
    round(test_positive_f1_v2, 4)
)


print("\nClassification Report:")

print(
    classification_report(
        y_test_v2,
        test_pred_v2,
        digits=4
    )
)


print("Confusion Matrix:")

print(
    confusion_matrix(
        y_test_v2,
        test_pred_v2
    )
)

Test shape: (935093, 20)

===== FINAL TEST RESULTS — V2 =====
Fixed threshold: 0.53
ROC-AUC: 0.5362
PR-AUC: 0.1729
Macro F1: 0.5161
Disengaged-class F1: 0.1887

Classification Report:
              precision    recall  f1-score   support

           0     0.8487    0.8384    0.8435    788700
           1     0.1829    0.1949    0.1887    146393

    accuracy                         0.7376    935093
   macro avg     0.5158    0.5166    0.5161    935093
weighted avg     0.7445    0.7376    0.7410    935093

Confusion Matrix:
[[661229 127471]
 [117867  28526]]


In [28]:
# =========================================================
# STEP 17 — SAVE BEHAVIOUR DISENGAGEMENT V2 MODEL
# =========================================================

import json

BEHAVIOR_DISENGAGEMENT_MODEL_PATH = (
    "/kaggle/working/"
    "mindbridge_disengagement_behavior_model_v2.json"
)

BEHAVIOR_DISENGAGEMENT_INFO_PATH = (
    "/kaggle/working/"
    "disengagement_behavior_model_v2_info.json"
)


# Save trained XGBoost model
disengagement_behavior_model_v2.save_model(
    BEHAVIOR_DISENGAGEMENT_MODEL_PATH
)


behavior_disengagement_metadata = {

    "model_name":
        "MindBridge Behaviour Disengagement Baseline V2",

    "model_type":
        "XGBoost",

    "dataset":
        "ASSISTments 2012-2013 with Affect",

    "target":
        "Behavioural disengagement proxy",

    "target_rule":
        "Average_confidence(BORED) >= 0.50",

    "threshold":
        round(
            float(
                FINAL_BEHAVIOR_DISENGAGEMENT_THRESHOLD
            ),
            2
        ),

    "features":
        behavior_features_v2,

    "learner_history": {
        "window_size": 20,
        "minimum_previous_interactions": 5,
        "baseline_coverage_percent": 96.32
    },

    "class_weight":
        round(
            float(scale_pos_weight_v2),
            4
        ),

    "validation_metrics": {
        "roc_auc": 0.5360,
        "pr_auc": 0.1712,
        "macro_f1": 0.5153,
        "disengaged_f1": 0.1830
    },

    "test_metrics": {
        "roc_auc": 0.5362,
        "pr_auc": 0.1729,
        "macro_f1": 0.5161,
        "disengaged_precision": 0.1829,
        "disengaged_recall": 0.1949,
        "disengaged_f1": 0.1887
    },

    "comparison_with_v1": {
        "v1_validation_roc_auc": 0.5290,
        "v2_validation_roc_auc": 0.5360,

        "v1_validation_pr_auc": 0.1679,
        "v2_validation_pr_auc": 0.1712,

        "v1_validation_macro_f1": 0.5126,
        "v2_validation_macro_f1": 0.5153,

        "conclusion":
            "V2 produced a small improvement using learner-specific temporal features."
    },

    "note":
        (
            "ASSISTments does not provide a direct disengagement label. "
            "High boredom confidence is used as a behavioural disengagement proxy. "
            "The model has limited standalone predictive performance and is "
            "intended for multimodal fusion."
        )
}


with open(
    BEHAVIOR_DISENGAGEMENT_INFO_PATH,
    "w"
) as f:

    json.dump(
        behavior_disengagement_metadata,
        f,
        indent=4
    )


print(
    "Behaviour disengagement model saved ✅"
)

print(
    BEHAVIOR_DISENGAGEMENT_MODEL_PATH
)

print(
    "\nMetadata saved ✅"
)

print(
    BEHAVIOR_DISENGAGEMENT_INFO_PATH
)

Behaviour disengagement model saved ✅
/kaggle/working/mindbridge_disengagement_behavior_model_v2.json

Metadata saved ✅
/kaggle/working/disengagement_behavior_model_v2_info.json


In [29]:
# =========================================================
# STEP 18A — DISENGAGEMENT FUSION HELPERS
# =========================================================

import numpy as np


def threshold_centered_evidence(prob, threshold):
    """
    Maps a model probability so that its selected decision threshold
    becomes 0.5 evidence.
    """

    prob = np.asarray(prob, dtype=float)

    return np.where(
        prob <= threshold,

        0.5 * (
            prob /
            (threshold + 1e-8)
        ),

        0.5 + 0.5 * (
            (prob - threshold) /
            (1 - threshold + 1e-8)
        )
    )


def reliability_aware_fusion(
    visual_evidence,
    behavior_evidence,
    visual_reliability,
    behavior_reliability
):

    visual_evidence = np.asarray(
        visual_evidence,
        dtype=float
    )

    behavior_evidence = np.asarray(
        behavior_evidence,
        dtype=float
    )

    visual_reliability = np.clip(
        np.asarray(
            visual_reliability,
            dtype=float
        ),
        0,
        1
    )

    behavior_reliability = np.clip(
        np.asarray(
            behavior_reliability,
            dtype=float
        ),
        0,
        1
    )


    total_reliability = (
        visual_reliability +
        behavior_reliability +
        1e-8
    )


    visual_weight = (
        visual_reliability /
        total_reliability
    )

    behavior_weight = (
        behavior_reliability /
        total_reliability
    )


    fused_evidence = (
        visual_weight *
        visual_evidence
        +
        behavior_weight *
        behavior_evidence
    )


    disagreement = np.abs(
        visual_evidence -
        behavior_evidence
    )


    shared_reliability = np.minimum(
        visual_reliability,
        behavior_reliability
    )


    conflict_strength = (
        disagreement *
        shared_reliability
    )


    weighted_reliability = (
        visual_weight *
        visual_reliability
        +
        behavior_weight *
        behavior_reliability
    )


    fusion_confidence = (
        weighted_reliability *
        (1 - conflict_strength)
    )


    fusion_confidence = np.clip(
        fusion_confidence,
        0,
        1
    )


    return {
        "visual_weight":
            visual_weight,

        "behavior_weight":
            behavior_weight,

        "fused_evidence":
            fused_evidence,

        "disagreement":
            disagreement,

        "conflict_strength":
            conflict_strength,

        "fusion_confidence":
            fusion_confidence
    }


print("Disengagement fusion helpers ready ✅")

Disengagement fusion helpers ready ✅


In [30]:
# =========================================================
# STEP 18B-1 — VISUAL RELIABILITY FOR DISENGAGEMENT
# =========================================================

import numpy as np

# Train-only pose thresholds
pose_cols = [
    "mean_abs_pitch",
    "mean_abs_yaw",
    "mean_abs_roll"
]

pose_thresholds = {}

for col in pose_cols:

    q75 = train_disengagement[col].quantile(0.75)
    q95 = train_disengagement[col].quantile(0.95)

    pose_thresholds[col] = {
        "q75": float(q75),
        "q95": float(q95)
    }

    print(
        col,
        "q75 =", round(q75, 4),
        "q95 =", round(q95, 4)
    )


def add_visual_reliability(df):

    df = df.copy()

    qualities = []

    for col in pose_cols:

        q75 = pose_thresholds[col]["q75"]
        q95 = pose_thresholds[col]["q95"]

        quality = 1 - np.clip(
            (df[col] - q75) /
            (q95 - q75 + 1e-8),
            0,
            1
        )

        qualities.append(quality)

    pose_quality = (
        qualities[0] +
        qualities[1] +
        qualities[2]
    ) / 3.0

    df["pose_quality"] = pose_quality

    df["visual_reliability"] = np.clip(

        df["face_visibility_ratio"] *
        (
            0.5 +
            0.5 * pose_quality
        ),

        0,
        1
    )

    return df


train_disengagement = add_visual_reliability(
    train_disengagement
)

val_disengagement = add_visual_reliability(
    val_disengagement
)

test_disengagement = add_visual_reliability(
    test_disengagement
)


for name, df in [
    ("TRAIN", train_disengagement),
    ("VALIDATION", val_disengagement),
    ("TEST", test_disengagement)
]:

    r = df["visual_reliability"]

    print(f"\n===== {name} VISUAL RELIABILITY =====")

    print(r.describe())

    print(
        "< 0.80:",
        int((r < 0.80).sum())
    )

    print(
        "< 0.60:",
        int((r < 0.60).sum())
    )

mean_abs_pitch q75 = 10.6009 q95 = 18.1916
mean_abs_yaw q75 = 11.3656 q95 = 20.6081
mean_abs_roll q75 = 6.3687 q95 = 11.2431

===== TRAIN VISUAL RELIABILITY =====
count    5358.000000
mean        0.938349
std         0.082258
min         0.479895
25%         0.887678
50%         0.984701
75%         1.000000
max         1.000000
Name: visual_reliability, dtype: float64
< 0.80: 367
< 0.60: 4

===== VALIDATION VISUAL RELIABILITY =====
count    1429.000000
mean        0.915530
std         0.099225
min         0.521562
25%         0.833333
50%         0.957961
75%         1.000000
max         1.000000
Name: visual_reliability, dtype: float64
< 0.80: 174
< 0.60: 7

===== TEST VISUAL RELIABILITY =====
count    1784.000000
mean        0.915424
std         0.112625
min         0.500000
25%         0.838230
50%         0.982184
75%         1.000000
max         1.000000
Name: visual_reliability, dtype: float64
< 0.80: 317
< 0.60: 13


In [31]:
# =========================================================
# STEP 18B-2 — BEHAVIOUR RELIABILITY FOR DISENGAGEMENT
# =========================================================

# Train-only response-time limits
train_rt = behavior_disengagement_df.loc[
    train_mask_v2,
    "ms_first_response"
].astype(float)

train_rt = train_rt[
    train_rt >= 0
]

rt_q01 = train_rt.quantile(0.01)
rt_q99 = train_rt.quantile(0.99)

print(
    "Train RT q01:",
    round(rt_q01, 2)
)

print(
    "Train RT q99:",
    round(rt_q99, 2)
)


# Start from fully reliable
behavior_reliability = np.ones(
    len(behavior_disengagement_df),
    dtype=np.float32
)


# Missing bottom_hint
behavior_reliability -= (
    behavior_disengagement_df[
        "bottom_hint_missing"
    ].to_numpy(dtype=np.float32)
    * 0.10
)


# Invalid negative timing
invalid_timing = (
    (
        behavior_disengagement_df[
            "ms_first_response"
        ] < 0
    )
    |
    (
        behavior_disengagement_df[
            "overlap_time"
        ] < 0
    )
)

behavior_reliability[
    invalid_timing.to_numpy()
] -= 0.50


# Extremely short response
extreme_short = (
    behavior_disengagement_df[
        "ms_first_response"
    ] < rt_q01
)

behavior_reliability[
    extreme_short.to_numpy()
] -= 0.10


# Extremely long response
extreme_long = (
    behavior_disengagement_df[
        "ms_first_response"
    ] > rt_q99
)

behavior_reliability[
    extreme_long.to_numpy()
] -= 0.10


# Keep base quality inside 0–1
behavior_reliability = np.clip(
    behavior_reliability,
    0,
    1
)


# Add learner-history confidence
history_factor = (
    0.70 +
    0.30 *
    behavior_disengagement_df[
        "history_confidence"
    ].to_numpy(dtype=np.float32)
)


behavior_reliability = (
    behavior_reliability *
    history_factor
)


behavior_reliability = np.clip(
    behavior_reliability,
    0,
    1
)


behavior_disengagement_df[
    "behavior_reliability"
] = behavior_reliability


print("\n===== BEHAVIOUR RELIABILITY =====")

print(
    behavior_disengagement_df[
        "behavior_reliability"
    ].describe()
)

print(
    "\n< 0.80:",
    int(
        (
            behavior_disengagement_df[
                "behavior_reliability"
            ] < 0.80
        ).sum()
    )
)

print(
    "< 0.60:",
    int(
        (
            behavior_disengagement_df[
                "behavior_reliability"
            ] < 0.60
        ).sum()
    )
)

Train RT q01: 1969.0
Train RT q99: 377326.28

===== BEHAVIOUR RELIABILITY =====
count    6.123270e+06
mean     9.756913e-01
std      6.503548e-02
min      4.000000e-01
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: behavior_reliability, dtype: float64

< 0.80: 315677
< 0.60: 36


In [32]:
# =========================================================
# STEP 18C — TEST RELIABILITY-AWARE DISENGAGEMENT FUSION
# =========================================================

VISUAL_DISENGAGEMENT_THRESHOLD = 0.54
BEHAVIOR_DISENGAGEMENT_THRESHOLD = 0.53


# ---------------------------------------------------------
# Synthetic synchronized windows
#
# IMPORTANT:
# These are ONLY prototype examples.
# DAiSEE + ASSISTments rows are NOT being fake-merged.
# ---------------------------------------------------------

synthetic_visual_prob = np.array([
    0.30,   # low disengagement evidence
    0.65,   # both modalities suggest disengagement
    0.80,   # visual strong but unreliable
    0.75,   # visual strong
    0.82    # both strong
])

synthetic_behavior_prob = np.array([
    0.35,
    0.68,
    0.75,
    0.35,   # modalities disagree
    0.80
])


synthetic_visual_rel = np.array([
    0.95,
    0.90,
    0.20,   # unreliable camera
    0.90,
    0.85
])

synthetic_behavior_rel = np.array([
    0.95,
    0.95,
    0.95,
    0.90,
    0.95
])


# ---------------------------------------------------------
# 1. Threshold-centered evidence
# ---------------------------------------------------------

visual_evidence = threshold_centered_evidence(
    synthetic_visual_prob,
    VISUAL_DISENGAGEMENT_THRESHOLD
)

behavior_evidence = threshold_centered_evidence(
    synthetic_behavior_prob,
    BEHAVIOR_DISENGAGEMENT_THRESHOLD
)


# ---------------------------------------------------------
# 2. Reliability-aware fusion
# ---------------------------------------------------------

fusion_test = reliability_aware_fusion(
    visual_evidence,
    behavior_evidence,
    synthetic_visual_rel,
    synthetic_behavior_rel
)


# ---------------------------------------------------------
# 3. Display results
# ---------------------------------------------------------

fusion_demo = pd.DataFrame({

    "visual_prob":
        synthetic_visual_prob,

    "behavior_prob":
        synthetic_behavior_prob,

    "visual_evidence":
        visual_evidence,

    "behavior_evidence":
        behavior_evidence,

    "visual_reliability":
        synthetic_visual_rel,

    "behavior_reliability":
        synthetic_behavior_rel,

    "visual_weight":
        fusion_test["visual_weight"],

    "behavior_weight":
        fusion_test["behavior_weight"],

    "fused_evidence":
        fusion_test["fused_evidence"],

    "disagreement":
        fusion_test["disagreement"],

    "fusion_confidence":
        fusion_test["fusion_confidence"]
})


display(
    fusion_demo.round(4)
)

,visual_prob,behavior_prob,visual_evidence,behavior_evidence,visual_reliability,behavior_reliability,visual_weight,behavior_weight,fused_evidence,disagreement,fusion_confidence
0,0.30,0.35,0.2778,0.3302,0.95,0.95,0.5000,0.5000,0.3040,0.0524,0.9027
1,0.65,0.68,0.6196,0.6596,0.90,0.95,0.4865,0.5135,0.6401,0.0400,0.8923
2,0.80,0.75,0.7826,0.7340,0.20,0.95,0.1739,0.8261,0.7425,0.0486,0.8116
3,0.75,0.35,0.7283,0.3302,0.90,0.90,0.5000,0.5000,0.5292,0.3981,0.5776
4,0.82,0.80,0.8043,0.7872,0.85,0.95,0.4722,0.5278,0.7953,0.0171,0.8896


In [33]:
# =========================================================
# STEP 19 — TEMPORAL DISENGAGEMENT DECISION
# =========================================================

import numpy as np
import pandas as pd


def temporal_disengagement_decision(
    fused_evidence,
    fusion_confidence,
    window_size=3,
    disengagement_threshold=0.50,
    intervention_threshold=0.60,
    min_persistent_windows=2,
    min_confidence=0.50
):

    fused_evidence = np.asarray(
        fused_evidence,
        dtype=float
    )

    fusion_confidence = np.asarray(
        fusion_confidence,
        dtype=float
    )

    results = []

    for i in range(len(fused_evidence)):

        start = max(
            0,
            i - window_size + 1
        )

        recent_evidence = fused_evidence[
            start:i + 1
        ]

        recent_confidence = fusion_confidence[
            start:i + 1
        ]


        # -----------------------------------------
        # Confidence-weighted temporal score
        # -----------------------------------------

        confidence_sum = (
            recent_confidence.sum()
        )

        if confidence_sum > 0:

            temporal_score = (
                np.sum(
                    recent_evidence *
                    recent_confidence
                )
                /
                confidence_sum
            )

        else:

            temporal_score = 0.5


        avg_confidence = (
            recent_confidence.mean()
        )


        # -----------------------------------------
        # Count persistent disengaged windows
        # -----------------------------------------

        persistent_count = np.sum(
            (
                recent_evidence >=
                disengagement_threshold
            )
            &
            (
                recent_confidence >=
                min_confidence
            )
        )


        current_evidence = (
            fused_evidence[i]
        )


        # -----------------------------------------
        # State decision
        # -----------------------------------------

        if avg_confidence < 0.40:

            state = "uncertain"
            support_need = "collect_more_evidence"


        elif (
            current_evidence <
            disengagement_threshold
            and
            temporal_score >=
            disengagement_threshold
        ):

            state = "recovering_engagement"
            support_need = "monitor"


        elif (
            current_evidence >=
            disengagement_threshold
            and
            temporal_score >=
            intervention_threshold
            and
            persistent_count >=
            min_persistent_windows
        ):

            state = "persistent_disengagement"
            support_need = "request_reengagement_support"


        elif (
            current_evidence >=
            disengagement_threshold
        ):

            state = "transient_disengagement"
            support_need = "monitor"


        else:

            state = "engaged"
            support_need = "none"


        results.append({

            "temporal_score":
                temporal_score,

            "avg_confidence":
                avg_confidence,

            "persistent_count":
                int(persistent_count),

            "state":
                state,

            "support_need":
                support_need
        })


    return pd.DataFrame(results)


print(
    "Temporal disengagement logic ready ✅"
)

Temporal disengagement logic ready ✅


In [34]:
# =========================================================
# TEST — SAME LEARNER ACROSS 6 WINDOWS
# =========================================================

demo_disengagement_evidence = [
    0.30,
    0.55,
    0.66,
    0.72,
    0.76,
    0.40
]

demo_disengagement_confidence = [
    0.90,
    0.88,
    0.86,
    0.84,
    0.82,
    0.90
]


temporal_demo = temporal_disengagement_decision(

    fused_evidence=
        demo_disengagement_evidence,

    fusion_confidence=
        demo_disengagement_confidence,

    window_size=3,

    disengagement_threshold=0.50,

    intervention_threshold=0.60,

    min_persistent_windows=2,

    min_confidence=0.50
)


temporal_demo.insert(
    0,
    "fused_evidence",
    demo_disengagement_evidence
)

temporal_demo.insert(
    1,
    "fusion_confidence",
    demo_disengagement_confidence
)


display(
    temporal_demo.round(4)
)

,fused_evidence,fusion_confidence,temporal_score,avg_confidence,persistent_count,state,support_need
0,0.30,0.90,0.3000,0.9000,0,engaged,none
1,0.55,0.88,0.4236,0.8900,1,transient_disengagement,monitor
2,0.66,0.86,0.5006,0.8800,2,transient_disengagement,monitor
3,0.72,0.84,0.6420,0.8600,3,persistent_disengagement,request_reengagement_support
4,0.76,0.82,0.7125,0.8400,3,persistent_disengagement,request_reengagement_support
5,0.40,0.90,0.6203,0.8533,2,recovering_engagement,monitor


In [35]:
# =========================================================
# STEP 20 — COMPLETE DISENGAGEMENT ENGINE
# =========================================================

def mindbridge_disengagement_engine(
    visual_evidence,
    behavior_evidence,
    visual_reliability,
    behavior_reliability,
    window_size=3,
    disengagement_threshold=0.50,
    intervention_threshold=0.60
):

    # -----------------------------------------------------
    # 1. Reliability-aware fusion
    # -----------------------------------------------------

    fusion = reliability_aware_fusion(
        visual_evidence,
        behavior_evidence,
        visual_reliability,
        behavior_reliability
    )


    # -----------------------------------------------------
    # 2. Temporal persistence
    # -----------------------------------------------------

    temporal = temporal_disengagement_decision(

        fused_evidence=
            fusion["fused_evidence"],

        fusion_confidence=
            fusion["fusion_confidence"],

        window_size=
            window_size,

        disengagement_threshold=
            disengagement_threshold,

        intervention_threshold=
            intervention_threshold,

        min_persistent_windows=2,

        min_confidence=0.50
    )


    # -----------------------------------------------------
    # 3. Combined output
    # -----------------------------------------------------

    result = pd.DataFrame({

        "visual_evidence":
            visual_evidence,

        "behavior_evidence":
            behavior_evidence,

        "visual_reliability":
            visual_reliability,

        "behavior_reliability":
            behavior_reliability,

        "visual_weight":
            fusion["visual_weight"],

        "behavior_weight":
            fusion["behavior_weight"],

        "fused_evidence":
            fusion["fused_evidence"],

        "modality_disagreement":
            fusion["disagreement"],

        "fusion_confidence":
            fusion["fusion_confidence"]
    })


    result["temporal_score"] = (
        temporal["temporal_score"]
    )

    result["avg_confidence"] = (
        temporal["avg_confidence"]
    )

    result["persistent_count"] = (
        temporal["persistent_count"]
    )

    result["learner_state"] = (
        temporal["state"]
    )

    result["support_need"] = (
        temporal["support_need"]
    )


    return result


print("Full MindBridge Disengagement Engine ready ✅")

Full MindBridge Disengagement Engine ready ✅


In [36]:
# =========================================================
# TEST COMPLETE DISENGAGEMENT ENGINE
# =========================================================

visual_prob_demo = np.array([
    0.30,
    0.58,
    0.67,
    0.73,
    0.80,
    0.40
])

behavior_prob_demo = np.array([
    0.35,
    0.57,
    0.65,
    0.70,
    0.76,
    0.42
])


visual_rel_demo = np.array([
    0.95,
    0.90,
    0.88,
    0.85,
    0.80,
    0.90
])

behavior_rel_demo = np.array([
    0.95,
    0.95,
    0.95,
    0.95,
    0.95,
    0.95
])


# ---------------------------------------------------------
# Convert raw model probabilities → normalized evidence
# ---------------------------------------------------------

visual_evidence_demo = threshold_centered_evidence(
    visual_prob_demo,
    0.54
)

behavior_evidence_demo = threshold_centered_evidence(
    behavior_prob_demo,
    0.53
)


# ---------------------------------------------------------
# Run full engine
# ---------------------------------------------------------

full_disengagement_demo = (
    mindbridge_disengagement_engine(

        visual_evidence=
            visual_evidence_demo,

        behavior_evidence=
            behavior_evidence_demo,

        visual_reliability=
            visual_rel_demo,

        behavior_reliability=
            behavior_rel_demo
    )
)


display(
    full_disengagement_demo.round(4)
)

,visual_evidence,behavior_evidence,visual_reliability,behavior_reliability,visual_weight,behavior_weight,fused_evidence,modality_disagreement,fusion_confidence,temporal_score,avg_confidence,persistent_count,learner_state,support_need
0,0.2778,0.3302,0.95,0.95,0.5000,0.5000,0.3040,0.0524,0.9027,0.3040,0.9027,0,engaged,none
1,0.5435,0.5426,0.90,0.95,0.4865,0.5135,0.5430,0.0009,0.9249,0.4249,0.9138,1,transient_disengagement,monitor
2,0.6413,0.6277,0.88,0.95,0.4809,0.5191,0.6342,0.0136,0.9053,0.4943,0.9110,2,transient_disengagement,monitor
3,0.7065,0.6809,0.85,0.95,0.4722,0.5278,0.6930,0.0257,0.8831,0.6222,0.9044,3,persistent_disengagement,request_reengagement_support
4,0.7826,0.7447,0.80,0.95,0.4571,0.5429,0.7620,0.0379,0.8547,0.6952,0.8810,3,persistent_disengagement,request_reengagement_support
5,0.3704,0.3962,0.90,0.95,0.4865,0.5135,0.3836,0.0259,0.9041,0.6095,0.8806,2,recovering_engagement,monitor


In [37]:
# =========================================================
# STEP 21 — SAVE DISENGAGEMENT ENGINE CONFIG + SUMMARY
# =========================================================

import json


# ---------------------------------------------------------
# 1. Fusion / temporal configuration
# ---------------------------------------------------------

disengagement_config = {

    "system":
        "MindBridge Disengagement Detection Engine",

    "version":
        "prototype_v1",

    "visual_model": {
        "dataset": "DAiSEE",
        "target":
            "Low-engagement proxy",
        "target_rule":
            "Engagement <= 1",
        "threshold":
            0.54
    },

    "behavior_model": {
        "dataset":
            "ASSISTments 2012-2013 with Affect",
        "target":
            "Behavioural disengagement proxy",
        "target_rule":
            "Average_confidence(BORED) >= 0.50",
        "threshold":
            0.53
    },

    "reliability": {
        "visual":
            "Face visibility and head-pose quality",
        "behavior":
            (
                "Data quality, timing validity, "
                "and learner-history confidence"
            )
    },

    "learner_history": {
        "window_size":
            20,
        "minimum_history":
            5,
        "baseline_coverage_percent":
            96.32
    },

    "fusion": {
        "method":
            "Dynamic reliability-weighted fusion",
        "threshold_centering":
            True,
        "conflict_handling":
            True
    },

    "temporal": {
        "window_size":
            3,
        "disengagement_threshold":
            0.50,
        "intervention_threshold":
            0.60,
        "minimum_persistent_windows":
            2,
        "minimum_confidence":
            0.50
    },

    "states": [
        "engaged",
        "transient_disengagement",
        "persistent_disengagement",
        "recovering_engagement",
        "uncertain"
    ],

    "support_actions": [
        "none",
        "monitor",
        "request_reengagement_support",
        "collect_more_evidence"
    ],

    "validation_note":
        (
            "Fusion thresholds and temporal rules are "
            "prototype heuristics. Final calibration "
            "requires synchronized MindBridge pilot data."
        )
}


CONFIG_PATH = (
    "/kaggle/working/"
    "mindbridge_disengagement_fusion_config.json"
)

with open(CONFIG_PATH, "w") as f:
    json.dump(
        disengagement_config,
        f,
        indent=4
    )


# ---------------------------------------------------------
# 2. Research summary
# ---------------------------------------------------------

disengagement_summary = {

    "component":
        "MindBridge Reliability-Aware Disengagement Detection",

    "status":
        "core_prototype_complete",

    "visual_model": {

        "model":
            "XGBoost",

        "threshold":
            0.54,

        "test_metrics": {
            "roc_auc":
                0.6871,
            "pr_auc":
                0.1390,
            "macro_f1":
                0.5497,
            "precision_disengaged":
                0.1275,
            "recall_disengaged":
                0.4318,
            "f1_disengaged":
                0.1969
        }
    },

    "behavior_model": {

        "model":
            "XGBoost V2 with learner-specific temporal features",

        "threshold":
            0.53,

        "test_metrics": {
            "roc_auc":
                0.5362,
            "pr_auc":
                0.1729,
            "macro_f1":
                0.5161,
            "precision_disengaged":
                0.1829,
            "recall_disengaged":
                0.1949,
            "f1_disengaged":
                0.1887
        },

        "v1_vs_v2":
            (
                "V2 produced a small improvement "
                "over the static behavioural baseline."
            )
    },

    "fusion": {
        "reliability_aware":
            True,
        "conflict_aware":
            True,
        "temporal_persistence":
            True
    },

    "important_limitation":
        (
            "DAiSEE and ASSISTments are not synchronized "
            "multimodal datasets. Public-dataset rows were "
            "not artificially paired. Final multimodal "
            "evaluation requires synchronized pilot data."
        ),

    "final_note":
        (
            "The core disengagement prototype is complete, "
            "but the final multimodal system is not yet "
            "scientifically validated on synchronized learners."
        )
}


SUMMARY_PATH = (
    "/kaggle/working/"
    "mindbridge_disengagement_engine_summary.json"
)

with open(SUMMARY_PATH, "w") as f:
    json.dump(
        disengagement_summary,
        f,
        indent=4
    )


print("Disengagement fusion config saved ✅")
print(CONFIG_PATH)

print("\nDisengagement engine summary saved ✅")
print(SUMMARY_PATH)

Disengagement fusion config saved ✅
/kaggle/working/mindbridge_disengagement_fusion_config.json

Disengagement engine summary saved ✅
/kaggle/working/mindbridge_disengagement_engine_summary.json
